# 06 · Can the same review team flag more fraud amount?
Compare two queues for the validation window from notebook 02. Both use the same trained ANN and have exactly **K** review slots. Set K to the number of transactions flagged by the saved cutoff in notebook 03.

**Score order** takes the K highest scores. **Amount within high risk** takes the 2K highest scores, then fills K slots by descending transaction amount. The shortlist multiplier is fixed at two before inspecting results. Restricting the shortlist to high scores keeps the ANN involved without interpreting its class-weighted output as a calibrated fraud probability.

This retrospective experiment assumes a reviewer can rank the entire window as one batch. The measured amount is the face value of confirmed fraud sent for review. Actual loss and recovery are unknown.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection")
run_id = "20261121T162317186996Z"
checkpoint = project / "Artifacts" / f"ann_{run_id}.pt"
csv_path = project / "Dataset" / "creditcard.csv"

Mounted at /content/drive


## Reproduce the validation scores
Sort by time and use the middle 70%–85% interval from training. As in notebook 03, transactions with the same timestamp stay together. Original CSV record numbers provide a deterministic tie-breaker.

In [2]:
saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
features = saved["features"]
ann = nn.Sequential(nn.Linear(len(features), 64), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
ann.load_state_dict(saved["state_dict"])
ann.eval()  # Disable dropout while ranking transactions.

data = pd.read_csv(csv_path)
data.index = pd.RangeIndex(1, len(data) + 1, name="source_record")
data = data.sort_values("Time", kind="stable")
start, stop = data.Time.iloc[[int(.70 * len(data)), int(.85 * len(data))]]
window = data.loc[(data.Time >= start) & (data.Time < stop)]
scaled = (window[features].to_numpy() - np.asarray(saved["mean"])) / np.asarray(saved["scale"])
with torch.inference_mode():
    batches = torch.from_numpy(scaled.astype("float32")).split(4096)
    scores = torch.cat([ann(batch).flatten().sigmoid() for batch in batches]).numpy()
candidates = window[["Amount"]].assign(score=scores).reset_index()
k = int((scores >= saved["threshold"]).sum())
assert 0 < k <= len(window)
print(f"{len(window):,} validation records | {k} reviews available")

42,719 validation records | 42 reviews available


In [3]:
# Select both queues using only information available when a transaction is scored.
ranked = candidates.sort_values(["score", "source_record"], ascending=[False, True])
score_queue = ranked.head(k)
shortlist = ranked.head(2 * k)
amount_queue = shortlist.sort_values(["Amount", "score", "source_record"],
                                     ascending=[False, False, True]).head(k)

def record_ids(queue):
    ids = pd.Index(queue.source_record)
    assert ids.is_unique and len(ids) == k
    return ids

baseline = record_ids(score_queue)
challenger = record_ids(amount_queue)
assert set(baseline) == set(candidates.loc[candidates.score >= saved["threshold"], "source_record"])
assert challenger.isin(shortlist.source_record).all()

## Compare results at equal workload
Precision measures the share of reviewed records that were fraudulent. Count recall divides caught frauds by all frauds in the window. Amount recall divides the sum of caught fraud amounts by the sum of all fraud amounts in the window.

In [4]:
truth = window[["Class", "Amount"]]
all_frauds = truth.loc[truth.Class.eq(1)]

def evaluate(ids):
    reviewed = truth.loc[ids]
    caught = reviewed.loc[reviewed.Class.eq(1)]
    return {"Reviews": len(reviewed), "Frauds caught": len(caught),
            "False alerts": len(reviewed) - len(caught),
            "Precision": len(caught) / len(reviewed),
            "Count recall": len(caught) / len(all_frauds),
            "Fraud amount": caught.Amount.sum(),
            "Amount recall": caught.Amount.sum() / all_frauds.Amount.sum()}

comparison = pd.DataFrame([evaluate(baseline), evaluate(challenger)],
                          index=["Score order", "Amount within high risk"])
print(f"Validation frauds: {len(all_frauds)} | Total fraud amount: {all_frauds.Amount.sum():,.2f}")
print(comparison.round(4).to_string())

Validation frauds: 56 | Total fraud amount: 8,397.70
                         Reviews  Frauds caught  False alerts  Precision  Count recall  Fraud amount  Amount recall
Score order                   42             42             0     1.0000        0.7500       7983.85         0.9507
Amount within high risk       42             24            18     0.5714        0.4286       8214.64         0.9782


## Explain the difference using the swapped reviews
Transactions present in both queues have the same outcome under both policies. Only the added and removed records can explain a change in performance. Compare the number of frauds and their total amounts in each group.

In [5]:
groups = {"Kept by both": baseline.intersection(challenger),
          "Dropped by challenger": baseline.difference(challenger),
          "Added by challenger": challenger.difference(baseline)}
rows = []
for group, ids in groups.items():
    records = truth.loc[ids]
    frauds = records.loc[records.Class.eq(1)]
    rows.append({"Group": group, "Reviews": len(records), "Frauds": len(frauds),
                 "Legitimate": len(records) - len(frauds), "Fraud amount": frauds.Amount.sum()})

swaps = pd.DataFrame(rows).set_index("Group")
print(swaps.round(2).to_string())
delta = comparison.iloc[1] - comparison.iloc[0]
np.testing.assert_allclose(delta["Fraud amount"], swaps.iloc[2]["Fraud amount"] - swaps.iloc[1]["Fraud amount"])
print(f"Change at the same budget: {delta['Frauds caught']:+.0f} frauds, "
      f"{delta['False alerts']:+.0f} false alerts, {delta['Fraud amount']:+,.2f} fraud amount.")

                       Reviews  Frauds  Legitimate  Fraud amount
Group                                                           
Kept by both                22      22           0       7968.09
Dropped by challenger       20      20           0         15.76
Added by challenger         20       2          18        246.55
Change at the same budget: -18 frauds, +18 false alerts, +230.79 fraud amount.


## Operating decision
Keep score order as the default for a team prioritizing case detection. The amount-aware queue flagged 230.79 more in fraud amount while missing 18 additional frauds at the same workload. It displaced 20 confirmed frauds totaling only 15.76 and brought in two larger frauds totaling 246.55 plus 18 legitimate transactions. This tradeoff is meaningful only in the context of the team's objectives.

## Decision scope
The ANN checkpoint and original cutoff were chosen using this validation window, and the earlier test audit motivated the challenger. This comparison describes these validation transactions. A prospective comparison on a fresh time window would be needed to estimate the benefit of changing the operating policy.